# AI-Optimized Yield Advisor: Exploratory Data Analysis
**Verilumen Labs Pvt. Ltd. | AX7-SOC Chip-Test Benchmark**

This notebook provides interactive exploration and technical verification for:
1. STDF Wafer Sort Data & Stop-on-Fail Verification
2. Spatial Wafer Map Feature Extraction & Pattern Classification
3. Statistical Process Control (EWMA & CUSUM) for Yield Excursions
4. Multi-Factor Commonality Analysis for Root Cause Attribution
5. AEC-Q001 Dynamic Part Average Testing (DPAT) & Good-Die-in-Bad-Neighborhood (GDBN)
6. Test Time Optimization, Correlation Analysis & Rule-of-Three Risk Bound
7. Pre-Probe Yield Forecasting from Fab Metrology

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('.').resolve().parent))

import pandas as pd
import numpy as np
import config
from src.data_loader import DataLoader
from src.preprocessing import Preprocessor
from src.wafer_map import WaferMapClassifier
from src.excursion_detection import ExcursionDetector
from src.root_cause import RootCauseEngine
from src.latent_defect import LatentDefectEngine
from src.test_optimizer import TestTimeOptimizer
from src.yield_prediction import YieldPredictor

loader = DataLoader()
lots = loader.load_lots()
ws = loader.load_wafer_summary()
print(f'Total Lots: {len(lots)}, Total Wafers: {len(ws)}, Mean Final Yield: {ws.final_yield.mean()*100:.2f}%')

## 1. Wafer Map Spatial Feature Extraction
We extract concentric radial gradients, line projections, circular cluster densities, and reticle periodicity.

In [2]:
prep = Preprocessor(loader)
sample_wafer = loader.load_sort_die_results(lot_id='L262200', wafer_id=1)
feats = prep.extract_wafer_spatial_features(sample_wafer)
for k, v in feats.items():
    print(f'{k:25s}: {v}')

## 2. Statistical Commonality Analysis for Root Cause
Rank equipment and chambers by yield deficit using Fisher exact / Chi-Square p-values.

In [3]:
rc_engine = RootCauseEngine()
wafer_features = pd.read_parquet(config.PREPROCESSED_SUMMARY_PATH)
comm = rc_engine.run_commonality_analysis(wafer_features)
comm.head(5)[['factor_type', 'factor_value', 'affected_wafers', 'mean_yield_pct', 'yield_delta_pts', 'p_value']]

## 3. Latent Defect Screening (DPAT + GDBN) & Burn-in Correlation
Verify the 138x relative risk difference between screened outlier dies and baseline dies.

In [4]:
import json
with open(config.REPORTS_DIR / 'model_evaluation_report.json') as f:
    rep = json.load(f)
lat = rep['engine_4_latent_defect']
print(f'Flagged Outliers: {lat["flagged_screened_dies"]}')
print(f'Burn-in Fail Rate in Flagged: {lat["burnin_fail_rate_flagged_pct"]:.2f}% vs Unflagged: {lat["burnin_fail_rate_unflagged_pct"]:.4f}%')
print(f'Relative Risk Ratio: {lat["relative_risk_ratio"]}x, DPPM Reduction: {lat["dppm_reduction_pct"]}%, Yield Cost: {lat["yield_cost_pts"]} pts')